# Step 1 — Extract Dataset from JPEG Images

Reads camera frames from `extracted_images/` and extracts **6 features**
from the green line inside a perspective-correct trapezoid ROI.

**Output files (saved to `data/`):**
- `X_features.npy` — feature matrix, shape (N, 6): `[off_bot, off_mid, off_top, has_bot, has_mid, has_top]`
- `y_labels.npy`   — continuous steering targets computed by geometric labeling formula

## Pipeline summary

1. Apply trapezoid ROI (matches road perspective, measured from actual frames)
2. HSV green mask + morphological cleanup inside trapezoid
3. Sample line position at 3 heights using adaptive row scanning:
   - **bot**: scan upward from bottom edge → first row with green pixels (stop at 60%)
   - **mid**: fixed slice at 35% of trapezoid height
   - **top**: scan downward from top edge → first row with green pixels (stop at 35%)
4. Compute offsets relative to car center (693px, empirically measured)
5. Apply geometric labeling formula with case detection → SVR training target

## Features (6 total)

| # | Feature | Range | Meaning |
|---|---------|-------|---------|
| 1 | `off_bot` | -1 … +1 | line x at bottom of trapezoid relative to car center (0.0 if missing) |
| 2 | `off_mid` | -1 … +1 | line x at 35% height relative to car center (0.0 if missing) |
| 3 | `off_top` | -1 … +1 | line x at top of trapezoid relative to car center (0.0 if missing) |
| 4 | `has_bot` | 0 or 1 | 1 if bottom band was detected, 0 if missing |
| 5 | `has_mid` | 0 or 1 | 1 if middle band was detected, 0 if missing |
| 6 | `has_top` | 0 or 1 | 1 if top band was detected, 0 if missing |

The `has_*` flags tell the SVR which offsets are real vs filled zeros — preventing it from
misreading a missing band as "line is centered."

## Label formula

Weighted blend of off_bot/mid/top with case detection:
- **crossover** (bot and top on opposite sides): `0.40×bot + 0.45×mid + 0.15×top`
- **curve coming** (disagreement > 0.15): `0.40×bot + 0.35×mid + 0.25×top`
- **straight/drift**: `0.60×bot + 0.25×mid + 0.15×top`
- **missing bands**: weights shift to available bands


## 1. Imports and paths

**Set `IMAGES_DIR` to the folder containing your JPEG files.**

In [1]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

REPO_ROOT  = Path().resolve().parent
IMAGES_DIR = REPO_ROOT / "extracted_images"
DATA_DIR   = REPO_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

image_paths = sorted(
    list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.jpeg")) + list(IMAGES_DIR.glob("*.JPG"))
)
assert len(image_paths) > 0, f"No JPEG files found in:\n{IMAGES_DIR}"
print(f"Images directory : {IMAGES_DIR}")
print(f"Total images     : {len(image_paths)}")
print(f"First image      : {image_paths[0].name}")
print(f"Last  image      : {image_paths[-1].name}")


Images directory : D:\HSHL_resources_v2\6th_Semester\New folder\Autonomous_Systems_A_Lab_Group_4\extracted_images
Total images     : 88544
First image      : frame_000001.jpg
Last  image      : frame_044272.jpg


## 2. Feature + label pipeline


In [ ]:
# ── Constants (must match 00_visualize_pipeline.ipynb and my_line_follower.py) ──
# Tightened from [40,40,40] to reject low-saturation foliage and shadows.
LOWER_GREEN = np.array([45,  60,  60])
UPPER_GREEN = np.array([85, 255, 255])

# Trapezoid ROI (measured from actual frames, H=720, W=1280)
TRAP_BOT_Y   = 0.85
TRAP_TOP_Y   = 0.45
TRAP_BOT_X_L = 0.328
TRAP_BOT_X_R = 0.789
TRAP_TOP_X_L = 0.403
TRAP_TOP_X_R = 0.697

# Car center empirically measured from straight frames (frame 4802 → cx_bot=693px)
CAR_CENTER_X = 693  # px


def extract_features_and_label(img_bgr: np.ndarray):
    """
    Extract 6 features and compute geometric steering label from one frame.

    Features: [off_bot, off_mid, off_top, has_bot, has_mid, has_top]
      - off_*: line x at that height relative to car center, normalised [-1,1], 0.0 if missing
      - has_*: 1.0 if band detected, 0.0 if missing — tells SVR which offsets are real

    Returns (feat np.float32 shape (6,), label float) or (None, None).
    """
    H, W = img_bgr.shape[:2]
    half = W / 2.0

    # ── Trapezoid mask ──────────────────────────────────────────────────
    trap_pts = np.array([
        [int(TRAP_TOP_X_L * W), int(TRAP_TOP_Y * H)],
        [int(TRAP_TOP_X_R * W), int(TRAP_TOP_Y * H)],
        [int(TRAP_BOT_X_R * W), int(TRAP_BOT_Y * H)],
        [int(TRAP_BOT_X_L * W), int(TRAP_BOT_Y * H)],
    ], dtype=np.int32)

    tm = np.zeros((H, W), dtype=np.uint8)
    cv2.fillPoly(tm, [trap_pts], 255)

    # ── Green mask ──────────────────────────────────────────────────────
    hsv   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)
    k     = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    green = cv2.morphologyEx(green, cv2.MORPH_CLOSE, k)
    green = cv2.morphologyEx(green, cv2.MORPH_OPEN,  k)
    green = cv2.bitwise_and(green, tm)

    y_top_t = trap_pts[:, 1].min()
    y_bot_t = trap_pts[:, 1].max()
    gmask   = green[y_top_t:y_bot_t, :]
    rr, cc  = gmask.shape[:2]

    ys, xs = np.where(gmask > 0)
    if len(ys) < 20:
        return None, None

    # ── Band sampling ───────────────────────────────────────────────────
    def band_cx_adaptive(search_start, search_end, slice_half=0.07):
        step      = -1 if search_start > search_end else 1
        start_row = int(rr * search_start)
        end_row   = int(rr * search_end)
        for centre_row in range(start_row, end_row, step):
            lo  = max(0,  centre_row - int(rr * slice_half))
            hi  = min(rr, centre_row + int(rr * slice_half))
            idx = (ys >= lo) & (ys < hi)
            if idx.sum() >= 5:
                return float(xs[idx].mean())
        return None

    def band_cx_fixed(centre_frac, slice_half=0.07):
        lo  = int(rr * max(0.0, centre_frac - slice_half))
        hi  = int(rr * min(1.0, centre_frac + slice_half))
        if hi <= lo: hi = lo + 1
        idx = (ys >= lo) & (ys < hi)
        return float(xs[idx].mean()) if idx.sum() >= 3 else None

    cx_bot = band_cx_adaptive(0.99, 0.60)
    cx_mid = band_cx_fixed(0.35)
    cx_top = band_cx_adaptive(0.01, 0.35)

    off_bot = float(np.clip((cx_bot - CAR_CENTER_X) / half, -1.0, 1.0)) if cx_bot is not None else None
    off_mid = float(np.clip((cx_mid - CAR_CENTER_X) / half, -1.0, 1.0)) if cx_mid is not None else None
    off_top = float(np.clip((cx_top - CAR_CENTER_X) / half, -1.0, 1.0)) if cx_top is not None else None

    has_bot = off_bot is not None
    has_mid = off_mid is not None
    has_top = off_top is not None

    if not (has_bot or has_mid or has_top):
        return None, None

    # ── Geometric label formula ─────────────────────────────────────────
    crossover = has_bot and has_top and (off_bot * off_top < 0)

    if crossover:
        mid_val = off_mid if has_mid else 0.0
        label   = 0.40 * off_bot + 0.45 * mid_val + 0.15 * off_top

    elif has_bot and has_mid and has_top:
        disagreement = abs(off_top - off_bot)
        if disagreement > 0.15:
            label = 0.40 * off_bot + 0.35 * off_mid + 0.25 * off_top
        else:
            label = 0.60 * off_bot + 0.25 * off_mid + 0.15 * off_top

    elif has_bot and has_mid:
        label = 0.65 * off_bot + 0.35 * off_mid

    elif has_mid and has_top:
        label = 0.60 * off_mid + 0.40 * off_top

    elif has_mid:
        label = off_mid

    elif has_top:
        label = off_top

    else:
        label = off_bot

    label = float(np.clip(label, -1.0, 1.0))

    # ── 6-feature vector: offsets + validity flags ──────────────────────
    feat = np.array([
        off_bot if has_bot else 0.0,
        off_mid if has_mid else 0.0,
        off_top if has_top else 0.0,
        1.0 if has_bot else 0.0,
        1.0 if has_mid else 0.0,
        1.0 if has_top else 0.0,
    ], dtype=np.float32)

    return feat, label


print("Feature + label pipeline defined.")
print(f"HSV: lower={LOWER_GREEN.tolist()}  upper={UPPER_GREEN.tolist()}")
print(f"CAR_CENTER_X = {CAR_CENTER_X}px  |  trapezoid: bot_y={TRAP_BOT_Y} top_y={TRAP_TOP_Y}")
print("Features: [off_bot, off_mid, off_top, has_bot, has_mid, has_top]  (6 total)")
print("Label: geometric weighted blend with crossover/curve/drift case detection")


## 3. Sanity check â€” single frame

In [ ]:
sample_img = cv2.imread(str(image_paths[0]))
feat, label = extract_features_and_label(sample_img)

if feat is None:
    print("No green line detected in first frame — check IMAGES_DIR or HSV thresholds")
else:
    assert len(feat) == 6, f"Expected 6 features, got {len(feat)} — restart kernel and run all"
    direction = "LEFT" if label < -0.05 else "RIGHT" if label > 0.05 else "STRAIGHT"
    print(f"Feature vector : {feat}")
    print(f"off_bot={feat[0]:+.3f}  off_mid={feat[1]:+.3f}  off_top={feat[2]:+.3f}")
    print(f"has_bot={feat[3]:.0f}    has_mid={feat[4]:.0f}    has_top={feat[5]:.0f}")
    print(f"Label          : {label:+.4f}  ({direction})")

    plt.figure(figsize=(8, 4))
    plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Sample frame — label={label:+.3f} ({direction})")
    plt.axis("off")
    plt.show()


## 4. Extract features from all images

In [ ]:
SUBSAMPLE = 2   # 1 = every frame; 2 = every other frame

X_list, y_list = [], []
skipped = 0
paths_to_use = image_paths[::SUBSAMPLE]
print(f"Processing {len(paths_to_use)} images (SUBSAMPLE={SUBSAMPLE})...")

for idx, img_path in enumerate(paths_to_use):
    img = cv2.imread(str(img_path))
    if img is None:
        skipped += 1
        continue

    feat, label = extract_features_and_label(img)
    if feat is None:
        skipped += 1
        continue

    X_list.append(feat)
    y_list.append(label)

    if (idx + 1) % 5000 == 0:
        print(f"  {idx+1}/{len(paths_to_use)} processed  ({len(X_list)} extracted)")

print(f"\nDone.  Extracted: {len(X_list)}  |  skipped: {skipped}")


Processing 44272 images (SUBSAMPLE=2)...
  5000/44272 processed  (4923 extracted)


## 5. Build labels and inspect distribution

In [ ]:
X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)

print(f"Feature matrix X : {X.shape}  (N samples × 3 features)")
print(f"Labels y         : {y.shape}  (continuous steering targets)")
print(f"Label range      : [{y.min():+.3f}, {y.max():+.3f}]  mean={y.mean():+.3f}")
print()

# Label distribution
DEAD_BAND = 0.05
left     = (y < -DEAD_BAND).sum()
straight = ((y >= -DEAD_BAND) & (y <= DEAD_BAND)).sum()
right    = (y > DEAD_BAND).sum()
print(f"  LEFT     : {left:5d}  ({100*left/len(y):.1f}%)")
print(f"  STRAIGHT : {straight:5d}  ({100*straight/len(y):.1f}%)")
print(f"  RIGHT    : {right:5d}  ({100*right/len(y):.1f}%)")

# Label distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(y, bins=80, color="steelblue", edgecolor="none")
axes[0].axvline(-DEAD_BAND, color="red", linestyle="--", linewidth=1.5)
axes[0].axvline( DEAD_BAND, color="red", linestyle="--", linewidth=1.5)
axes[0].set_xlabel("steering label")
axes[0].set_ylabel("frame count")
axes[0].set_title("Label distribution")

# Feature distributions
feat_names = ["off_bot", "off_mid", "off_top"]
for i, name in enumerate(feat_names):
    axes[1].hist(X[:, i], bins=60, alpha=0.5, label=name, edgecolor="none")
axes[1].set_xlabel("offset value")
axes[1].set_ylabel("count")
axes[1].set_title("Feature distributions")
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Save to data/

In [ ]:
np.save(str(DATA_DIR / "X_features.npy"), X)
np.save(str(DATA_DIR / "y_labels.npy"),   y)

print("Saved:")
for f in sorted(DATA_DIR.glob("*.npy")):
    print(f"  {f.name}  —  shape={np.load(str(f)).shape}  ({f.stat().st_size // 1024} KB)")
print("\nDone. Run 02_train_svm.ipynb next.")
